# Phase 9 — Baseline Modelling Contract

Validate and freeze the metadata-only modelling input contract that downstream baseline model phases are allowed to consume.

**Phase 9 does not train a model, calculate model metrics, alter preprocessing artifacts, or overwrite Phase 8 outputs.**

## A. Notebook Setup

This notebook runs from a fresh kernel. It uses the same production functions as Phase 8 to prepare verified preprocessing outputs in memory, then passes those outputs into the Phase 9 contract gate.

In [11]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from urban_ops.eda.pipeline import load_eda_config
from urban_ops.eda.source import load_verified_split, resolve_split_run, verify_source_unchanged
from urban_ops.features.categorical_encoding import (
    fit_categorical_encoder,
    load_categorical_encoding_config,
    transform_split_categorical_encoder,
)
from urban_ops.features.categorical_missing import (
    load_categorical_missing_config,
    replace_split_categorical_missing,
)
from urban_ops.features.numeric_preprocessing import (
    fit_numeric_preprocessor,
    load_numeric_preprocessing_config,
    transform_split_numeric_preprocessor,
)
from urban_ops.features.policy import load_feature_policy
from urban_ops.features.preprocessing_composition import (
    build_preprocessing_composition,
    compose_split_preprocessing_blocks,
    load_preprocessing_composition_config,
)
from urban_ops.features.preprocessing_verification import (
    VerifiedPreprocessingContract,
    build_final_feature_schema_evidence,
    build_preprocessing_verification_evidence,
    build_training_feature_variance_evidence,
    load_preprocessing_verification_config,
    verify_preprocessing_contract,
)
from urban_ops.features.rare_unseen import (
    fit_rare_unseen_handler,
    load_rare_unseen_config,
    transform_split_rare_unseen,
)
from urban_ops.features.temporal import derive_split_temporal_features
from urban_ops.models.baseline_contract import (
    VerifiedBaselineModellingContract,
    build_baseline_modelling_contract_evidence,
    load_baseline_modelling_contract_config,
    verify_baseline_modelling_contract,
)

## B. Governed Configuration

All configuration is loaded from existing project files. The notebook does not edit these files.

In [12]:
EDA_CONFIG_PATH = PROJECT_ROOT / "configs/eda/resolution_risk.yaml"
POLICY_PATH = PROJECT_ROOT / "configs/features/resolution_risk_baseline.yaml"
MISSING_CONFIG_PATH = PROJECT_ROOT / "configs/features/resolution_risk_categorical_missing.yaml"
CARDINALITY_CONFIG_PATH = PROJECT_ROOT / "configs/features/resolution_risk_categorical_cardinality.yaml"
ENCODING_CONFIG_PATH = PROJECT_ROOT / "configs/features/resolution_risk_categorical_encoding.yaml"
NUMERIC_CONFIG_PATH = PROJECT_ROOT / "configs/features/resolution_risk_numeric_preprocessing.yaml"
COMPOSITION_CONFIG_PATH = PROJECT_ROOT / "configs/features/resolution_risk_preprocessing_composition.yaml"
VERIFICATION_CONFIG_PATH = PROJECT_ROOT / "configs/features/resolution_risk_preprocessing_verification.yaml"
MODELLING_CONTRACT_CONFIG_PATH = PROJECT_ROOT / "configs/models/resolution_risk_baseline_modelling_contract.yaml"

eda_config = load_eda_config(EDA_CONFIG_PATH)
policy = load_feature_policy(POLICY_PATH)
missing_config = load_categorical_missing_config(MISSING_CONFIG_PATH)
cardinality_config = load_rare_unseen_config(CARDINALITY_CONFIG_PATH, missing_config=missing_config)
encoding_config = load_categorical_encoding_config(ENCODING_CONFIG_PATH, cardinality_config=cardinality_config)
numeric_config = load_numeric_preprocessing_config(NUMERIC_CONFIG_PATH)
composition_config = load_preprocessing_composition_config(COMPOSITION_CONFIG_PATH)
verification_config = load_preprocessing_verification_config(VERIFICATION_CONFIG_PATH)
modelling_contract_config = load_baseline_modelling_contract_config(MODELLING_CONTRACT_CONFIG_PATH)

pd.Series(
    {
        "feature_policy_version": policy.policy_version,
        "phase_8_verification_status": verification_config.verification_status,
        "phase_9_contract_status": modelling_contract_config.contract_status,
        "required_splits": modelling_contract_config.required_splits,
    }
)

feature_policy_version                                 1
phase_8_verification_status           FROZEN_MODEL_READY
phase_9_contract_status            MODEL_INPUTS_VERIFIED
required_splits                (train, validation, test)
dtype: object

## C. Load Authoritative Chronological Splits

The source is the latest verified chronological split, not raw data. Rows are not sorted, repaired, or overwritten.

In [13]:
run_path = resolve_split_run(
    split_root=eda_config.split_root,
    latest_pointer=eda_config.latest_pointer,
)
source = load_verified_split(
    run_path=run_path,
    latest_pointer=eda_config.latest_pointer,
    required_completion_status=eda_config.required_completion_status,
    identifier_column=eda_config.identifier_column,
    target_column=eda_config.target_column,
    timestamp_column=eda_config.timestamp_column,
)
source_frames = {"train": source.train, "validation": source.validation, "test": source.test}
source_snapshots = {split: frame.copy(deep=True) for split, frame in source_frames.items()}

pd.DataFrame(
    [
        {
            "split": split,
            "row_count": len(frame),
            "target_name": eda_config.target_column,
            "identifier_name": eda_config.identifier_column,
            "timestamp_name": eda_config.timestamp_column,
            "timestamp_min": frame[eda_config.timestamp_column].min(),
            "timestamp_max": frame[eda_config.timestamp_column].max(),
        }
        for split, frame in source_frames.items()
    ]
)

,split,row_count,target_name,identifier_name,timestamp_name,timestamp_min,timestamp_max
0,train,23699,missed_resolution_target,unique_key,created_date,2024-01-01 07:58:15+00:00,2025-03-31 23:31:13+00:00
1,validation,6762,missed_resolution_target,unique_key,created_date,2025-04-01 01:22:19+00:00,2025-08-31 23:49:25+00:00
2,test,5499,missed_resolution_target,unique_key,created_date,2025-09-01 04:49:31+00:00,2025-12-31 20:39:09+00:00


## D. Rebuild Phase 8 Verified Outputs In Memory

This section reproduces the existing Phase 2-8 production path in memory so notebook 12 can run independently. It does not persist matrices, change preprocessing policy, or train a model.

In [14]:
derived_frames = derive_split_temporal_features(source_frames, policy=policy)
missing_frames = replace_split_categorical_missing(derived_frames, policy=policy, config=missing_config)
fitted_cardinality = fit_rare_unseen_handler(missing_frames["train"], policy=policy, config=cardinality_config)
cardinality_frames = transform_split_rare_unseen(missing_frames, fitted=fitted_cardinality, config=cardinality_config)
fitted_categorical = fit_categorical_encoder(
    cardinality_frames["train"],
    policy=policy,
    config=encoding_config,
    fitted_cardinality=fitted_cardinality,
)
categorical_matrices = transform_split_categorical_encoder(cardinality_frames, fitted=fitted_categorical, config=encoding_config)
fitted_numeric = fit_numeric_preprocessor(cardinality_frames["train"], policy=policy, config=numeric_config)
numeric_matrices = transform_split_numeric_preprocessor(cardinality_frames, fitted=fitted_numeric, config=numeric_config)
fitted_composition = build_preprocessing_composition(
    fitted_categorical=fitted_categorical,
    fitted_numeric=fitted_numeric,
    policy=policy,
    config=composition_config,
)
matrices = compose_split_preprocessing_blocks(
    categorical_matrices=categorical_matrices,
    numeric_matrices=numeric_matrices,
    fitted=fitted_composition,
)
targets = {split: frame[eda_config.target_column] for split, frame in source_frames.items()}
identifiers = {split: frame[eda_config.identifier_column] for split, frame in source_frames.items()}
timestamps = {split: frame[eda_config.timestamp_column] for split, frame in source_frames.items()}
feature_names_by_split = {
    split: fitted_composition.combined_feature_names
    for split in ("train", "validation", "test")
}

pd.Series(
    {
        "phase_4_cardinality_fingerprint": fitted_cardinality.fingerprint,
        "phase_5_encoder_fingerprint": fitted_categorical.fingerprint,
        "phase_6_numeric_fingerprint": fitted_numeric.fingerprint,
        "phase_7_composition_fingerprint": fitted_composition.fingerprint,
        "matrix_type": fitted_composition.matrix_type,
        "output_dtype": fitted_composition.output_dtype,
        "model_training_implemented": False,
    }
)

phase_4_cardinality_fingerprint    f0b35d15c9b84684182a935cd78562ab053b0e0e99b1ee...
phase_5_encoder_fingerprint        994e97e12f027bacd2e9a317f4313b50ea18c22dd9bd76...
phase_6_numeric_fingerprint        f0c4edde034d3503a76f92665f40f4ea95c9e840b8444c...
phase_7_composition_fingerprint    b569e4633cba9ff5892cc47164eb6184ede324fc02119d...
matrix_type                                                               csr_matrix
output_dtype                                                                 float64
model_training_implemented                                                     False
dtype: object

In [15]:
verified_preprocessing_contract = verify_preprocessing_contract(
    matrices=matrices,
    targets=targets,
    identifiers=identifiers,
    timestamps=timestamps,
    feature_names_by_split=feature_names_by_split,
    target_name=eda_config.target_column,
    identifier_name=eda_config.identifier_column,
    timestamp_name=eda_config.timestamp_column,
    policy=policy,
    fitted_cardinality=fitted_cardinality,
    fitted_categorical=fitted_categorical,
    fitted_numeric=fitted_numeric,
    fitted_composition=fitted_composition,
    config=verification_config,
)
reloaded_preprocessing_contract = VerifiedPreprocessingContract.from_dict(
    json.loads(json.dumps(verified_preprocessing_contract.to_dict()))
)

pd.Series(
    {
        "phase_8_status": verified_preprocessing_contract.verification_status,
        "phase_8_feature_count": verified_preprocessing_contract.combined_feature_count,
        "phase_8_schema_fingerprint": verified_preprocessing_contract.schema_fingerprint,
        "phase_8_contract_fingerprint": verified_preprocessing_contract.fingerprint,
        "phase_8_serialization_round_trip": reloaded_preprocessing_contract == verified_preprocessing_contract,
    }
)

phase_8_status                                                     FROZEN_MODEL_READY
phase_8_feature_count                                                               4
phase_8_schema_fingerprint          e9d9f5f655ef0b20876fa7576db8d3bdc18e12228264b0...
phase_8_contract_fingerprint        ff741ecded0a617d09bfd7405acc68c6e337909bbc43d2...
phase_8_serialization_round_trip                                                 True
dtype: object

## E. Frozen Feature Schema

Phase 8 remains the source of truth for feature names and order.

In [16]:
phase_8_feature_names = verified_preprocessing_contract.combined_feature_names
build_final_feature_schema_evidence(
    fitted_categorical=fitted_categorical,
    fitted_numeric=fitted_numeric,
    fitted_composition=fitted_composition,
)

,column_index,feature_name,source_branch,source_feature,preprocessing_stage
0,0,created_hour,numeric,created_hour,phase_6_pass_through
1,1,created_day_of_week,numeric,created_day_of_week,phase_6_pass_through
2,2,created_month,numeric,created_month,phase_6_pass_through
3,3,is_weekend,numeric,is_weekend,phase_6_pass_through


## F. Phase 8 Readiness Evidence

This evidence confirms the modelling objects passed to Phase 9 are the same object family verified by Phase 8.

In [17]:
build_preprocessing_verification_evidence(
    matrices=matrices,
    targets=targets,
    identifiers=identifiers,
    contract=verified_preprocessing_contract,
)

,split,row_count,feature_count,target_count,identifier_count,matrix_type,dtype,nnz,density,non_finite_count,...,target_null_count,target_positive_count,target_negative_count,target_positive_rate,identifier_null_count,identifier_duplicate_count,row_alignment_valid,schema_valid,leakage_free,status
0,train,23699,4,23699,23699,csr_matrix,float64,72407,0.763819,0,...,0,10907,12792,0.460230,0,0,True,True,True,PASS
1,validation,6762,4,6762,6762,csr_matrix,float64,20560,0.760130,0,...,0,2882,3880,0.426205,0,0,True,True,True,PASS
2,test,5499,4,5499,5499,csr_matrix,float64,16089,0.731451,0,...,0,1927,3572,0.350427,0,0,True,True,True,PASS


In [18]:
build_training_feature_variance_evidence(matrix=matrices["train"], feature_names=phase_8_feature_names)

,column_index,feature_name,minimum,maximum,unique_count,all_zero,zero_variance
0,0,created_hour,0.0,23.0,24,False,False
1,1,created_day_of_week,0.0,6.0,7,False,False
2,2,created_month,1.0,12.0,12,False,False
3,3,is_weekend,0.0,1.0,2,False,False


## G. Phase 9 Verification Gate

The verifier checks Phase 8 lineage, split presence, CSR/float64 matrix shape, finite values, y/id/timestamp alignment, chronological ordering, disjoint identifiers, target domain, train-only class metadata, and feature isolation.

In [19]:
baseline_modelling_contract = verify_baseline_modelling_contract(
    matrices=matrices,
    targets=targets,
    identifiers=identifiers,
    timestamps=timestamps,
    feature_names=phase_8_feature_names,
    phase_8_contract=verified_preprocessing_contract,
    config=modelling_contract_config,
    expected_phase_8_fingerprint=verified_preprocessing_contract.fingerprint,
    expected_schema_fingerprint=verified_preprocessing_contract.schema_fingerprint,
    policy=policy,
)
baseline_modelling_contract.to_dict()

{'contract_version': 1,
 'modelling_policy_version': 1,
 'phase_8_contract_fingerprint': 'ff741ecded0a617d09bfd7405acc68c6e337909bbc43d2715c68a5fd8f1e00e3',
 'ordered_feature_names': ['created_hour',
  'created_day_of_week',
  'created_month',
  'is_weekend'],
 'feature_count': 4,
 'matrix_type': 'csr_matrix',
 'matrix_dtype': 'float64',
 'target_name': 'missed_resolution_target',
 'identifier_name': 'unique_key',
 'chronology_name': 'created_date',
 'split_order': ['train', 'validation', 'test'],
 'row_counts_by_split': {'train': 23699, 'validation': 6762, 'test': 5499},
 'train_row_count': 23699,
 'train_positive_class_count': 10907,
 'train_negative_class_count': 12792,
 'train_positive_class_prevalence': 0.46023038946791006,
 'train_majority_class': 0,
 'train_target_classes': [0, 1],
 'preprocessing_schema_fingerprint': 'e9d9f5f655ef0b20876fa7576db8d3bdc18e12228264b08a75d2671596b0aaf5',
 'status': 'MODEL_INPUTS_VERIFIED'}

## H. Split And Train-Only Target Evidence

Validation and test labels are verified for structure only. Baseline-defining target metadata is train-only.

In [20]:
build_baseline_modelling_contract_evidence(baseline_modelling_contract)

,split,row_count,feature_count,matrix_type,matrix_dtype,target_name,identifier_name,chronology_name,preprocessing_schema_fingerprint,phase_8_contract_fingerprint,status
0,train,23699,4,csr_matrix,float64,missed_resolution_target,unique_key,created_date,e9d9f5f655ef0b20876fa7576db8d3bdc18e12228264b0...,ff741ecded0a617d09bfd7405acc68c6e337909bbc43d2...,MODEL_INPUTS_VERIFIED
1,validation,6762,4,csr_matrix,float64,missed_resolution_target,unique_key,created_date,e9d9f5f655ef0b20876fa7576db8d3bdc18e12228264b0...,ff741ecded0a617d09bfd7405acc68c6e337909bbc43d2...,MODEL_INPUTS_VERIFIED
2,test,5499,4,csr_matrix,float64,missed_resolution_target,unique_key,created_date,e9d9f5f655ef0b20876fa7576db8d3bdc18e12228264b0...,ff741ecded0a617d09bfd7405acc68c6e337909bbc43d2...,MODEL_INPUTS_VERIFIED


In [21]:
train_target_evidence = pd.Series(
    {
        "train_row_count": baseline_modelling_contract.train_row_count,
        "positive_class_count": baseline_modelling_contract.train_positive_class_count,
        "negative_class_count": baseline_modelling_contract.train_negative_class_count,
        "positive_class_prevalence": baseline_modelling_contract.train_positive_class_prevalence,
        "majority_class": baseline_modelling_contract.train_majority_class,
    }
)
train_target_evidence

train_row_count              23699.00000
positive_class_count         10907.00000
negative_class_count         12792.00000
positive_class_prevalence        0.46023
majority_class                   0.00000
dtype: float64

## I. Fingerprint And Serialization Evidence

The contract fingerprint depends on metadata and Phase 8 lineage, not matrix payloads.

In [22]:
reloaded_baseline_modelling_contract = VerifiedBaselineModellingContract.from_dict(
    json.loads(json.dumps(baseline_modelling_contract.to_dict()))
)
repeated_baseline_modelling_contract = verify_baseline_modelling_contract(
    matrices=matrices,
    targets=targets,
    identifiers=identifiers,
    timestamps=timestamps,
    feature_names=phase_8_feature_names,
    phase_8_contract=verified_preprocessing_contract,
    config=modelling_contract_config,
    expected_phase_8_fingerprint=verified_preprocessing_contract.fingerprint,
    expected_schema_fingerprint=verified_preprocessing_contract.schema_fingerprint,
    policy=policy,
)

pd.Series(
    {
        "phase_8_contract_fingerprint": baseline_modelling_contract.phase_8_contract_fingerprint,
        "preprocessing_schema_fingerprint": baseline_modelling_contract.preprocessing_schema_fingerprint,
        "baseline_modelling_contract_fingerprint": baseline_modelling_contract.fingerprint,
        "serialization_round_trip": reloaded_baseline_modelling_contract == baseline_modelling_contract,
        "deterministic_fingerprint": repeated_baseline_modelling_contract.fingerprint == baseline_modelling_contract.fingerprint,
    }
)

phase_8_contract_fingerprint               ff741ecded0a617d09bfd7405acc68c6e337909bbc43d2...
preprocessing_schema_fingerprint           e9d9f5f655ef0b20876fa7576db8d3bdc18e12228264b0...
baseline_modelling_contract_fingerprint    d6620d99301f80884487140bde487479f58fe72c1e2093...
serialization_round_trip                                                                True
deterministic_fingerprint                                                               True
dtype: object

## J. Immutability And Scope Evidence

The notebook verifies that source splits remain unchanged and that Phase 9 stayed outside model training and evaluation.

In [23]:
source_artifacts_unchanged = True
for split in ("train", "validation", "test"):
    pd.testing.assert_frame_equal(source_frames[split], source_snapshots[split])
verify_source_unchanged(source)

forbidden_feature_names = {
    eda_config.target_column,
    eda_config.identifier_column,
    eda_config.timestamp_column,
    "borough",
    "location_type",
    "incident_zip",
    "latitude",
    "longitude",
    "closed_date",
    "due_date",
    "status",
}

pd.Series(
    {
        "source_artifacts_unchanged": source_artifacts_unchanged,
        "forbidden_features_absent": not forbidden_feature_names.intersection(
            baseline_modelling_contract.ordered_feature_names
        ),
        "unique_key_in_X": False,
        "created_date_in_X": False,
        "model_training_implemented": False,
        "model_evaluation_implemented": False,
        "preprocessing_artifacts_overwritten": False,
    }
)

source_artifacts_unchanged              True
forbidden_features_absent               True
unique_key_in_X                        False
created_date_in_X                      False
model_training_implemented             False
model_evaluation_implemented           False
preprocessing_artifacts_overwritten    False
dtype: bool

## K. Phase 9 Completion Decision

Phase 9 is complete only when the modelling inputs remain exactly aligned with the Phase 8 frozen preprocessing contract and the final status is `MODEL_INPUTS_VERIFIED`.

In [24]:
phase_9_complete = baseline_modelling_contract.status == "MODEL_INPUTS_VERIFIED"
pd.Series(
    {
        "phase": "Phase 9 — Baseline Modelling Contract",
        "phase_8_status": verified_preprocessing_contract.verification_status,
        "phase_9_status": baseline_modelling_contract.status,
        "feature_count": baseline_modelling_contract.feature_count,
        "feature_names": ", ".join(baseline_modelling_contract.ordered_feature_names),
        "split_order": baseline_modelling_contract.split_order,
        "phase_9_complete": phase_9_complete,
        "model_training_implemented": False,
        "model_evaluation_implemented": False,
        "preprocessing_modified": False,
    }
)

phase                                       Phase 9 — Baseline Modelling Contract
phase_8_status                                                 FROZEN_MODEL_READY
phase_9_status                                              MODEL_INPUTS_VERIFIED
feature_count                                                                   4
feature_names                   created_hour, created_day_of_week, created_mon...
split_order                                             (train, validation, test)
phase_9_complete                                                             True
model_training_implemented                                                  False
model_evaluation_implemented                                                False
preprocessing_modified                                                      False
dtype: object

## STOP — Baseline Training Boundary

**Phase 9 — Baseline Modelling Contract: COMPLETE**

**Status: MODEL_INPUTS_VERIFIED**

Next phase: **Phase 10 — Baseline Model Training**.